In [6]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from typing import TypedDict, Literal

from pydantic import BaseModel, Field

from utils.std_model import base_model

llm = base_model()


class State(TypedDict):
    topic: str
    content: str
    standard: str
    feedback: str
    good: str


class FeedBack(BaseModel):
    feedback: str = Field(
        description="需要根据标准来指定需要调整的内容和方向",
    )
    good: Literal["好", "不好"] = Field(
        description="枚举值，只有 “好” 和 “不好”"
    )


def llm_call(state: State):
    feedback = state.get("feedback", "")
    resp = llm.invoke(f"请根据主题[{state['topic']}]和反馈[{feedback}]生成一个故事")
    return {
        "content": resp.content
    }


def feedback_call(state: State):
    llm_so = llm.with_structured_output(FeedBack)
    resp = llm_so.invoke(f"请根据标准[{state['standard']}]来评估故事[{state['content']}]是否符合标准")
    return {
        "good": resp.good,
        "feedback": resp.feedback,
    }


def feedback_after_route(state: State):
    if state["good"] == "好":
        return END
    elif state["good"] == "不好":
        return feedback_call.__name__
    else:
        raise Exception


graph_builder = StateGraph(State)

graph_builder.add_node(llm_call.__name__, llm_call)
graph_builder.add_node(feedback_call.__name__, feedback_call)

graph_builder.add_edge(START, llm_call.__name__)
graph_builder.add_edge(llm_call.__name__, feedback_call.__name__)
graph_builder.add_conditional_edges(feedback_call.__name__, feedback_after_route, [llm_call.__name__, END])

graph = graph_builder.compile()
resp = graph.invoke({
    "topic": "遥知盼盼，独立西风",
    "standard": "故事需要包含一个明确的冲突和转折",
})




{'topic': '遥知盼盼，独立西风',
 'content': '边塞的风，总是比别处更硬一些。\n\n塞上秋来，草枯鹰疾，黄云压城。这座叫怀远的关隘已经安静了整整三年。三年前的那场大战，将军率三千铁骑出关，回来的人不到一半。这一仗，镇北关稳稳拿住了，敌军退到了阴山以北，此后很久没有再南下。可大帅钟衍，也没回来。\n\n据说他断后的时候，被流矢所伤，跌入了冰河之中。尸骨无存。\n\n消息传回京城的时候，已经是深秋。奉命传讯的斥候马都跑死了两匹，一路烟尘滚进城去，马蹄踏过朱雀街的青石板，惊得行人纷纷避让。\n\n盼盼就是在那个时候第一次听说丈夫的名字与“殉国”二字摆在一起的。\n\n她那时不过十八岁，嫁过去刚满一年。整整一年里，钟衍在家待的日子加起来不到半个月，她甚至没能记住他长什么样。只记得迎亲那日他掀开盖头，烛火底下匆匆看了她一眼，说了句什么。\n\n后来她回想了很多次，始终想不起来那句话究竟是什么。\n\n就像整个人还没反应过来，就成了寡妇。\n\n婆婆哭得几乎要断气，公公一夜之间白了头。家里的老仆人们唉声叹气，看她的眼神里满是同情，仿佛她是一件刚拆了封就被闲置的贵重物件。\n\n而盼盼没有哭。\n\n她安安静静地帮着料理完了丧事，给丈夫立了衣冠冢，便在每日黄昏时分走出家门，走到城西的望归台上去。\n\n望归台其实不是什么台，就是城墙西角的一处高高凸起的土坡，据说是前朝一位守将的妻子日日登高盼夫归的地方。后来那位守将回来了，妻子却已经哭瞎了眼睛。人们觉得这地方不吉利，平时没什么人去，倒是便宜了盼盼。\n\n她每天都要去那里站上一阵子。有时候是一炷香的工夫，有时候要到天色彻底暗下来，巡城的士卒举着火把从墙根底下过，看见她高高的剪影，便知道是钟家的儿媳又来望归了。\n\n这事传出去之后，府上的老嬷嬷劝过她几次，说少奶奶年纪轻轻，总往那种高台上跑，惹人闲话。婆婆更是直接多了，拉着她的手掉眼泪：“衍儿已经不在了，你还年轻，往后总得……总得往前看。”\n\n盼盼听懂了婆婆话里未说出口的意思——你改嫁，我不拦你。\n\n她只是笑着摇摇头，第二天黄昏照样出门。\n\n其实盼盼自己也说不清楚，她为什么要去。\n\n说思念吧，她与钟衍拢共没见过几面，谈不上什么刻骨铭心的情意。说执念吧，她也不是那种非要守节立坊的烈女。她只是觉得，那座台子望出去的方向，正好是他出关的方向。\n